In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.stats import beta
from scipy.special import digamma, polygamma
from GASModels.distributions.gb2_log_link import GB2LogLinkDistribution
from GASModels.distributions.za_gb2_log_link import ZAGB2LogLinkDistribution

In [2]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import os

In [3]:
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from itertools import product


def time_series_cv_score(X, y, model, initial_train_size, horizon=1):
    """
    Expanding window CV (Hyndman-style).
    
    Parameters
    ----------
    X : array-like
    y : array-like
    model : sklearn-style regressor
    initial_train_size : int
        Number of initial observations used for first training window
    horizon : int
        Forecast horizon (default = 1)
        
    Returns
    -------
    float
        RMSE averaged across folds
    """
    
    n = len(y)
    errors = []
    
    for t in range(initial_train_size, n - horizon + 1, 365):
        
        X_train_cv = X[:t]
        y_train_cv = y[:t]
        
        X_valid_cv = X[t:t+horizon]
        y_valid_cv = y[t:t+horizon]
        
        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_valid_cv)
        
        rmse = np.sqrt(mean_squared_error(y_valid_cv, y_pred))
        errors.append(rmse)
    
    return np.mean(errors)


def tune_xgb_time_series(X_train, y_train, param_grid,
                         initial_train_size,
                         horizon=1):
    
    best_score = np.inf
    best_params = None
    
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    
    for combination in product(*values):
        
        params = dict(zip(keys, combination))
        
        model = XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            **params
        )
        
        score = time_series_cv_score(
            X_train,
            y_train,
            model,
            initial_train_size=initial_train_size,
            horizon=horizon
        )
        
        print(f"Params: {params}, RMSE: {score:.5f}")
        
        if score < best_score:
            best_score = score
            best_params = params
    
    print("\nBest parameters:", best_params)
    print("Best CV RMSE:", best_score)
    
    return best_params

In [20]:
INPUT = r"C:\\Users\\ilang\\OneDrive\\Documentos\\Ilan\\academia\\dissertação\\data\\inmet\\inmet 250126\\TSLab input"

train = [x for x in os.listdir(INPUT) if x.endswith("_train.csv")]
test = [x for x in os.listdir(INPUT) if x.endswith("_test.csv")]

dfs_train = [pd.read_csv(os.path.join(INPUT, train[i]), sep=",") for i,_ in enumerate(train)]
dfs_test = [pd.read_csv(os.path.join(INPUT, test[i]), sep=",")  for i,_ in enumerate(train)]

In [5]:
param_grid = {
    "n_estimators": [500, 800, 1000],
    "max_depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.05]
}

In [6]:
best_fit = {k: [] for k in train}
best_fit

{'BELO HORIZONTE_train.csv': [],
 'CRUZEIRO DO SUL (ACRE)_train.csv': [],
 'GARANHUNS (PERNAMBUCO)_train.csv': [],
 'MACEIÓ_train.csv': [],
 'MANAUS_train.csv': [],
 'RIO DE JANEIRO (INMET)_train.csv': [],
 'SALVADOR_train.csv': [],
 'SÃO PAULO_train.csv': []}

In [ ]:
for i,_ in enumerate(dfs_train):
    
    print(train[i], ":")
    
    y = dfs_train[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].dropna().values[365:]

    X = pd.DataFrame(
        index = dfs_train[i]['Data Medicao'].values,
        data = {
            'x1': dfs_train[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].shift(1).values,
            'x2': dfs_train[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].shift(2).values,
            'x3': dfs_train[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].shift(3).values,
            'x_seas': dfs_train[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].shift(365).values
        }
    ).dropna()
    
    best_params = tune_xgb_time_series(
        X,
        y,
        param_grid,
        initial_train_size=365*5        ,  # example: first 2 years for training
        horizon=1
    )
    
    best_fit[train[i]].append(best_params)

BELO HORIZONTE_train.csv :
Params: {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.01}, RMSE: 7.90093
Params: {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05}, RMSE: 8.25070
Params: {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.01}, RMSE: 8.15968
Params: {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05}, RMSE: 7.10683
Params: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01}, RMSE: 8.24814
Params: {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05}, RMSE: 8.23577
Params: {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.01}, RMSE: 8.70758
Params: {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.05}, RMSE: 8.71007
Params: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.01}, RMSE: 8.61838
Params: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05}, RMSE: 8.12989
Params: {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.01}, RMSE: 7.76860
Params: {'n_estimators': 800, 'max_depth': 4, 'le

In [8]:
best_fit

{'BELO HORIZONTE_train.csv': [{'n_estimators': 1000,
   'max_depth': 5,
   'learning_rate': 0.05}],
 'CRUZEIRO DO SUL (ACRE)_train.csv': [{'n_estimators': 1000,
   'max_depth': 7,
   'learning_rate': 0.05}],
 'GARANHUNS (PERNAMBUCO)_train.csv': [{'n_estimators': 500,
   'max_depth': 8,
   'learning_rate': 0.05}],
 'MACEIÓ_train.csv': [{'n_estimators': 800,
   'max_depth': 6,
   'learning_rate': 0.05}],
 'MANAUS_train.csv': [{'n_estimators': 500,
   'max_depth': 7,
   'learning_rate': 0.01}],
 'RIO DE JANEIRO (INMET)_train.csv': [{'n_estimators': 500,
   'max_depth': 8,
   'learning_rate': 0.05}],
 'SALVADOR_train.csv': [{'n_estimators': 1000,
   'max_depth': 4,
   'learning_rate': 0.05}],
 'SÃO PAULO_train.csv': [{'n_estimators': 500,
   'max_depth': 7,
   'learning_rate': 0.05}]}

In [30]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

horizons = [1,2,5,10,30,60,90,182,365,540,730]

rmse_table = {}

models = []

# create models
for k, v in best_fit.items():
    models.append(XGBRegressor(objective="reg:squarederror",
                               random_state=42,
                               **v[0]))

for i, model in enumerate(models):

    df = dfs_train[i]
    df_test = dfs_test[i]

    series_name = test[i].split("_")[0]

    y_series = df['PRECIPITACAO TOTAL, DIARIO(mm)']

    data = pd.DataFrame({
        'y': y_series,
        'x1': y_series.shift(1),
        'x2': y_series.shift(2),
        'x3': y_series.shift(3),
        'x_seas': y_series.shift(365)
    }).dropna()

    X = data[['x1','x2','x3','x_seas']]
    y = data['y'].values

    # fit model
    model.fit(X, y)

    # recursive forecasting
    history = list(y_series.dropna().values)
    preds = []

    for t in range(730):

        x_new = pd.DataFrame({
            'x1':[history[-1]],
            'x2':[history[-2]],
            'x3':[history[-3]],
            'x_seas':[history[-365]]
        })

        yhat = model.predict(x_new)[0]

        preds.append(yhat)
        history.append(yhat)

    preds = np.array(preds)

    y_true = df_test['PRECIPITACAO TOTAL, DIARIO(mm)'].values[:730]

    rmse_h = {}

    for h in horizons:
        rmse = np.sqrt(mean_squared_error(y_true[:h], preds[:h]))
        rmse_h[h] = rmse

    rmse_table[series_name] = rmse_h


rmse_df = pd.DataFrame(rmse_table).T

In [32]:
summary_row = {}

for h in horizons:
    
    vals = rmse_df[h].astype(float).values
    
    mean = np.mean(vals)
    minv = np.min(vals)
    maxv = np.max(vals)

    summary_row[h] = f"\\textbf{{{mean:.2f} ({minv:.2f}--{maxv:.2f})}}"

rmse_df.loc["Mean"] = summary_row

In [33]:
latex_table = rmse_df.to_latex(
    escape=False,
    caption="Out-of-sample prediction RMSE for XGBoost for different $h$",
    label="tab:xgb_rmse"
)

print(latex_table)

\begin{table}
\caption{Out-of-sample prediction RMSE for XGBoost for different $h$}
\label{tab:xgb_rmse}
\begin{tabular}{llllllllllll}
\toprule
 & 1 & 2 & 5 & 10 & 30 & 60 & 90 & 182 & 365 & 540 & 730 \\
\midrule
BELO HORIZONTE & 38.600192 & 29.394713 & 21.976238 & 21.146744 & 16.676237 & 17.966445 & 17.636236 & 17.929043 & 16.894418 & 18.986375 & 19.445927 \\
CRUZEIRO DO SUL (ACRE) & 8.099682 & 17.286140 & 13.308555 & 20.616011 & 17.813822 & 19.328468 & 20.208749 & 16.883550 & 14.705702 & 15.278317 & 15.067765 \\
GARANHUNS (PERNAMBUCO) & 1.094643 & 3.154418 & 2.814845 & 4.820050 & 5.855726 & 5.976442 & 5.663982 & 11.021426 & 9.382982 & 9.406266 & 8.813234 \\
MACEIÓ & 5.858015 & 29.174278 & 19.392677 & 21.653089 & 32.471505 & 24.488347 & 21.301493 & 17.222600 & 15.638986 & 16.158700 & 16.175448 \\
MANAUS & 2.943332 & 3.028919 & 11.724565 & 15.359075 & 16.597278 & 21.066168 & 19.205193 & 18.792273 & 16.668548 & 17.590691 & 17.292064 \\
RIO DE JANEIRO (INMET) & 0.344251 & 9.474552 & 7.55

In [18]:
dfs_test[1]

,index,Data Medicao,"PRECIPITACAO TOTAL, DIARIO(mm)",Ano,smoothed
0,4778,2013-01-30,0.7,2013,10.476760
1,4779,2013-01-31,1.4,2013,10.651832
2,4780,2013-02-01,0.0,2013,10.805947
3,4781,2013-02-02,0.0,2013,10.933380
4,4782,2013-02-03,9.2,2013,11.029447
...,...,...,...,...,...
1090,5868,2016-01-25,0.0,2016,9.429878
1091,5869,2016-01-26,0.0,2016,9.580364
1092,5870,2016-01-27,5.8,2016,9.754425
1093,5871,2016-01-28,0.0,2016,9.943837


In [22]:
for i,_ in enumerate(all_forecasts):
    df = dfs_test[i].copy()
    mse = mean_squared_error(df['PRECIPITACAO TOTAL, DIARIO(mm)'].values, all_forecasts[i])
    print(mse)

378.1440855438815
223.2954957304885
62.94988471500033
245.6676195305063
285.95325307294246
88.26759503551271
214.18841027603085
233.76631602159165


SDM

In [39]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

horizons = [1,2,5,10,30,60,90,182,365,540,730]

forecast_mean = pd.read_csv("forecast_mean.csv")

rmse_table = {}

for i, col in enumerate(forecast_mean.drop('Unnamed: 0', axis=1).columns):

    series_name = col
    preds = forecast_mean[col].values[:730]

    y_true = dfs_test[i]['PRECIPITACAO TOTAL, DIARIO(mm)'].values[:730]

    rmse_h = {}

    for h in horizons:
        rmse = np.sqrt(mean_squared_error(y_true[:h], preds[:h]))
        rmse_h[h] = rmse

    rmse_table[series_name] = rmse_h


rmse_df = pd.DataFrame(rmse_table).T

In [40]:
summary_row = {}

for h in horizons:

    vals = rmse_df[h].astype(float).values

    mean = np.mean(vals)
    minv = np.min(vals)
    maxv = np.max(vals)

    summary_row[h] = f"\\textbf{{{mean:.2f} ({minv:.2f}--{maxv:.2f})}}"

rmse_df.loc["\\textbf{Mean}"] = summary_row

In [41]:
latex_table = rmse_df.to_latex(
    escape=False,
    caption="Out-of-sample prediction RMSE for the score-driven zero-augmented GB2 model for different $h$",
    label="tab:gasgb2_rmse"
)

print(latex_table)

\begin{table}
\caption{Out-of-sample prediction RMSE for the score-driven zero-augmented GB2 model for different $h$}
\label{tab:gasgb2_rmse}
\begin{tabular}{llllllllllll}
\toprule
 & 1 & 2 & 5 & 10 & 30 & 60 & 90 & 182 & 365 & 540 & 730 \\
\midrule
BELO HORIZONTE & 0.427128 & 3.591978 & 17.226937 & 17.026293 & 15.600935 & 15.665838 & 14.341940 & 10.847947 & 31.843127 & 47.065683 & 409.460681 \\
CRUZEIRO DO SUL (ACRE) & 0.450798 & 0.352210 & 3.467762 & 22.955570 & 19.525399 & 19.279818 & 20.036803 & 15.965245 & 14.536609 & 14.368312 & 20.946716 \\
GARANHUNS (PERNAMBUCO) & 0.724543 & 0.848949 & 0.976064 & 1.093337 & 1.827400 & 1.789905 & 2.315742 & 10.186565 & 8.604911 & 9.984438 & 2856.650402 \\
MACEIÓ & 11.515533 & 11.543202 & 10.193574 & 20.087617 & 30.928591 & 22.847355 & 21.647914 & 16.902504 & 40.075028 & 1071.818068 & 4106.295427 \\
MANAUS & 9.372754 & 7.015314 & 9.525900 & 14.100577 & 16.737285 & 20.202813 & 18.574336 & 16.314431 & 14.518935 & 18.067140 & 24.046226 \\
RIO DE JAN

LSTM

In [42]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


def train_lstm_fold(X_train, y_train, X_val, y_val, params):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = LSTMModel(
        input_size=X_train.shape[2],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"]
    ).to(device)
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"]  # L2 shrinkage
    )
    
    criterion = nn.MSELoss()
    
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                      torch.tensor(y_train, dtype=torch.float32)),
        batch_size=params["batch_size"],
        shuffle=False
    )
    
    best_loss = np.inf
    patience = 20
    counter = 0
    
    for epoch in range(params["max_epochs"]):
        
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
        
        # validation
        model.eval()
        with torch.no_grad():
            Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
            yv = torch.tensor(y_val, dtype=torch.float32).to(device)
            val_preds = model(Xv).squeeze()
            val_loss = criterion(val_preds, yv).item()
        
        # early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break
    
    return np.sqrt(best_loss)

In [43]:
def rolling_cv_lstm(X, y, initial_train_size, param_grid):
    
    best_score = np.inf
    best_params = None
    
    for params in param_grid:
        
        fold_errors = []
        
        for t in range(initial_train_size, len(y) - 1, 365):
            
            X_train_cv = X[:t]
            y_train_cv = y[:t]
            
            X_val_cv = X[t:t+1]
            y_val_cv = y[t:t+1]
            
            rmse = train_lstm_fold(
                X_train_cv,
                y_train_cv,
                X_val_cv,
                y_val_cv,
                params
            )
            
            fold_errors.append(rmse)
        
        avg_rmse = np.mean(fold_errors)
        print(f"Params: {params}, RMSE: {avg_rmse:.5f}")
        
        if avg_rmse < best_score:
            best_score = avg_rmse
            best_params = params
    
    print("\nBest Params:", best_params)
    return best_params

In [44]:
param_grid = [
    {
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.1,
        "learning_rate": 0.01,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "max_epochs": 200
    },
    {
        "hidden_size": 16,
        "num_layers": 2,
        "dropout": 0.2,
        "learning_rate": 0.005,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "max_epochs": 200
    }
]

In [45]:
best_params_lstm = {k: [] for k in train}

for i,_ in enumerate(dfs_train):
    
    print(train[i], ":")
    
    y_series = df['PRECIPITACAO TOTAL, DIARIO(mm)']

    data = pd.DataFrame({
        'y': y_series,
        'lag1': y_series.shift(1),
        'lag2': y_series.shift(2),
        'lag3': y_series.shift(3),
        'lag365': y_series.shift(365)
    }).dropna()

    X = data[['lag1','lag2','lag3','lag365']].values
    y = data['y'].values
    
    X = X.reshape(X.shape[0], X.shape[1], 1)
    
    split = int(len(X) * 0.8)

    X_train = X[:split]
    X_val   = X[split:]

    y_train = y[:split]
    y_val   = y[split:]
    
    results = []

    X_train = np.array(X_train).reshape(len(X_train), 4, 1)
    X_val   = np.array(X_val).reshape(len(X_val), 4, 1)

    for params in param_grid:
        
        rmse = train_lstm_fold(
            X_train,
            y_train,
            X_val,
            y_val,
            params
        )
        
        results.append((params, rmse))

    best_params, best_rmse = min(results, key=lambda x: x[1])

    print("Best parameters:", best_params)
    print("Best RMSE:", best_rmse)
    
    best_params_lstm[train[i]].append(best_params)

BELO HORIZONTE_train.csv :
Best parameters: {'hidden_size': 16, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.005, 'weight_decay': 0.0001, 'batch_size': 64, 'max_epochs': 200}
Best RMSE: 10.997331815848383
CRUZEIRO DO SUL (ACRE)_train.csv :
Best parameters: {'hidden_size': 16, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.005, 'weight_decay': 0.0001, 'batch_size': 64, 'max_epochs': 200}
Best RMSE: 10.977728405050772
GARANHUNS (PERNAMBUCO)_train.csv :
Best parameters: {'hidden_size': 16, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.005, 'weight_decay': 0.0001, 'batch_size': 64, 'max_epochs': 200}
Best RMSE: 10.965825834959983
MACEIÓ_train.csv :
Best parameters: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.01, 'weight_decay': 0.0001, 'batch_size': 64, 'max_epochs': 200}
Best RMSE: 10.984988286355485
MANAUS_train.csv :
Best parameters: {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.01, 'weight_decay': 0.0001, 'batch_si

In [50]:
best_params_lstm

{'BELO HORIZONTE_train.csv': [{'hidden_size': 16,
   'num_layers': 2,
   'dropout': 0.2,
   'learning_rate': 0.005,
   'weight_decay': 0.0001,
   'batch_size': 64,
   'max_epochs': 200}],
 'CRUZEIRO DO SUL (ACRE)_train.csv': [{'hidden_size': 16,
   'num_layers': 2,
   'dropout': 0.2,
   'learning_rate': 0.005,
   'weight_decay': 0.0001,
   'batch_size': 64,
   'max_epochs': 200}],
 'GARANHUNS (PERNAMBUCO)_train.csv': [{'hidden_size': 16,
   'num_layers': 2,
   'dropout': 0.2,
   'learning_rate': 0.005,
   'weight_decay': 0.0001,
   'batch_size': 64,
   'max_epochs': 200}],
 'MACEIÓ_train.csv': [{'hidden_size': 32,
   'num_layers': 1,
   'dropout': 0.1,
   'learning_rate': 0.01,
   'weight_decay': 0.0001,
   'batch_size': 64,
   'max_epochs': 200}],
 'MANAUS_train.csv': [{'hidden_size': 32,
   'num_layers': 1,
   'dropout': 0.1,
   'learning_rate': 0.01,
   'weight_decay': 0.0001,
   'batch_size': 64,
   'max_epochs': 200}],
 'RIO DE JANEIRO (INMET)_train.csv': [{'hidden_size': 32,
   '

In [52]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error

horizon = 730
horizons = [1,2,5,10,30,60,90,182,365,540,730]

rmse_table = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i, df in enumerate(dfs_train):

    df_test = dfs_test[i]
    series_name = test[i].split("_")[0]

    y_series = df['PRECIPITACAO TOTAL, DIARIO(mm)']

    # --- build lag dataset (same as XGBoost) ---
    
    data = pd.DataFrame({
        'y': y_series,
        'lag1': y_series.shift(1).values,
        'lag2': y_series.shift(2).values,
        'lag3': y_series.shift(3).values,
        'lag365': y_series.shift(365).values
    }).dropna()

    X = data[['lag1','lag2','lag3','lag365']].values
    y = data['y'].values

    # reshape for LSTM
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # --- train final model ---
    
    model = LSTMModel(
        input_size=1,
        hidden_size=best_params_lstm[train[i]][0]["hidden_size"],
        num_layers=best_params_lstm[train[i]][0]["num_layers"],
        dropout=best_params_lstm[train[i]][0]["dropout"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_params_lstm[train[i]][0]["learning_rate"],
        weight_decay=best_params_lstm[train[i]][0]["weight_decay"]
    )

    criterion = torch.nn.MSELoss()

    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )

    loader = DataLoader(
        dataset,
        batch_size=best_params_lstm[train[i]][0]["batch_size"],
        shuffle=False
    )

    model.train()

    for epoch in range(best_params_lstm[train[i]][0]["max_epochs"]):

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            preds = model(xb).squeeze()

            loss = criterion(preds, yb)

            loss.backward()

            optimizer.step()

    # --- recursive forecasting ---

    history = list(y_series.dropna().values)
    preds = []

    model.eval()

    with torch.no_grad():

        for t in range(horizon):

            x_new = np.array([
                history[-1],
                history[-2],
                history[-3],
                history[-365]
            ])

            x_new = x_new.reshape(1,4,1)

            x_tensor = torch.tensor(x_new, dtype=torch.float32).to(device)

            yhat = model(x_tensor).cpu().numpy()[0,0]

            preds.append(yhat)
            history.append(yhat)

    preds = np.array(preds)

    y_true = df_test['PRECIPITACAO TOTAL, DIARIO(mm)'].values[:730]

    rmse_h = {}

    for h in horizons:

        rmse = np.sqrt(mean_squared_error(y_true[:h], preds[:h]))

        rmse_h[h] = rmse

    rmse_table[series_name] = rmse_h

c:\Users\ilang\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\loss.py:616: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [54]:
rmse_df = pd.DataFrame(rmse_table).T
summary_row = {}

for h in horizons:

    vals = rmse_df[h].astype(float).values

    mean = np.mean(vals)
    minv = np.min(vals)
    maxv = np.max(vals)

    summary_row[h] = f"\\textbf{{{mean:.2f} ({minv:.2f}--{maxv:.2f})}}"

rmse_df.loc["\\textbf{Mean}"] = summary_row
latex_table = rmse_df.to_latex(
    escape=False,
    caption="Out-of-sample prediction RMSE for the LSTM model for different $h$",
    label="tab:lstm_rmse"
)

print(latex_table)

\begin{table}
\caption{Out-of-sample prediction RMSE for the LSTM model for different $h$}
\label{tab:lstm_rmse}
\begin{tabular}{llllllllllll}
\toprule
 & 1 & 2 & 5 & 10 & 30 & 60 & 90 & 182 & 365 & 540 & 730 \\
\midrule
BELO HORIZONTE & 3.413806 & 8.202605 & 20.992977 & 18.641880 & 15.457663 & 15.086154 & 14.289753 & 11.560967 & 12.537315 & 12.630450 & 13.136498 \\
CRUZEIRO DO SUL (ACRE) & 7.063441 & 9.666379 & 8.415552 & 21.229658 & 17.669855 & 17.689541 & 19.001365 & 15.859674 & 14.485932 & 14.279896 & 14.151923 \\
GARANHUNS (PERNAMBUCO) & 1.135471 & 1.361727 & 2.975232 & 3.080509 & 3.587315 & 3.650680 & 3.774681 & 10.397523 & 8.507610 & 7.665805 & 6.819093 \\
MACEIÓ & 7.992040 & 17.084492 & 12.574712 & 19.205582 & 31.219782 & 27.952576 & 36.190395 & 41.709100 & 46.732958 & 41.657700 & 36.175856 \\
MANAUS & 6.342939 & 6.401745 & 12.371070 & 16.831803 & 16.676972 & 20.248821 & 19.404532 & 17.814390 & 16.481527 & 18.005583 & 18.220907 \\
RIO DE JANEIRO (INMET) & 0.251821 & 5.994937 & 